# 01. ReAct 循环

ReAct = **Rea**soning + **Act**ing。核心思想：思考和行动交替进行，行动的结果（Observation）会成为下一轮思考的输入。

对比纯 Chain-of-Thought（只思考不行动），ReAct 的优势在于：
- 当模型缺乏知识时，可以主动搜索
- 当推理出错时，可以用计算器验证
- Observation 提供真实世界反馈，防止幻觉扩散

> **检查点**：能解释为什么「Observation → 下一轮 Thought」是 ReAct 区别于 CoT 的关键。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()


🌐 API 模式 — model=deepseek-v4-flash  base_url=https://api.deepseek.com


In [2]:
# ReAct 三阶段：Thought → Action → Observation
from dataclasses import dataclass, field

@dataclass
class ReActAgent:
    goal: str
    trace: list = field(default_factory=list)
    
    def think(self) -> str:
        """基于当前状态决定行动"""
        if not self.trace:
            return "search"
        last_obs = self.trace[-1].get("observation", "")
        if "answer" in last_obs.lower() or len(self.trace) >= 3:
            return "done"
        return "search"
    
    def act(self, action: str):
        """执行工具调用"""
        if action == "search":
            kb = {"python": "Python is a high-level language. Creator: Guido van Rossum. Year: 1991.",
                  "agent": "AI agent = model + tools + loop + state."}
            for k, v in kb.items():
                if k in self.goal.lower():
                    return v
            return "No info found."
        return "unknown action"

agent = ReActAgent(goal="learn about python")
for step in range(1, 6):
    thought = agent.think()
    print(f"--- Step {step} ---")
    print(f"  Thought: I should {thought}")
    
    if thought == "done":
        print(f"  Action: DONE")
        break
    
    observation = agent.act(thought)
    agent.trace.append({"thought": thought, "observation": observation})
    print(f"  Action: {thought}()")
    print(f"  Observation: {observation[:80]}")

print(f"\nTotal steps: {len(agent.trace)}")

--- Step 1 ---
  Thought: I should search
  Action: search()
  Observation: Python is a high-level language. Creator: Guido van Rossum. Year: 1991.
--- Step 2 ---
  Thought: I should search
  Action: search()
  Observation: Python is a high-level language. Creator: Guido van Rossum. Year: 1991.
--- Step 3 ---
  Thought: I should search
  Action: search()
  Observation: Python is a high-level language. Creator: Guido van Rossum. Year: 1991.
--- Step 4 ---
  Thought: I should done
  Action: DONE

Total steps: 3


## ReAct vs CoT 对比

| 维度 | Chain-of-Thought | ReAct |
|------|-----------------|-------|
| 信息源 | 仅模型内部知识 | 模型知识 + 外部工具 |
| 幻觉风险 | 高（无法验证） | 低（Observation 提供 grounding） |
| 适用场景 | 纯推理（数学、逻辑） | 需要外部信息（搜索、计算、API） |
| 调用次数 | 1 次 | N 次（loop） |

> **检查点**：能举例说明一个场景，CoT 会失败但 ReAct 能成功。

In [3]:
# 运行配套 demo 脚本
from pathlib import Path
import subprocess, sys

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent

SCRIPT = ROOT / 'agents' / 'paradigms' / 'scripts' / 'react_loop_demo.py'
result = subprocess.run(
    [sys.executable, str(SCRIPT), '--scenario', 'search', '--max-steps', '3', '--verbose'],
    capture_output=True, text=True
)
print(result.stdout)


--- Step 1 ---
  Thought: [toy] I should search
  Action: search('learn')
  Observation: No results for 'learn'

--- Step 2 ---
  Thought: [toy] I should search
  Action: search('learn')
  Observation: No results for 'learn'

--- Step 3 ---
  Thought: [toy] I should done
  Action: DONE

Final: steps=3, done=True, mode=toy



## 真实 API 实验

用真实 LLM 跑 ReAct 循环（`react_loop_demo.py --use-api`），对比 toy 规则引擎的轨迹差异。


In [4]:
# 真实 LLM 驱动的 ReAct 循环
SCRIPT = ROOT / 'agents' / 'paradigms' / 'scripts' / 'react_loop_demo.py'
import subprocess, sys
result = subprocess.run([sys.executable, str(SCRIPT), '--scenario', 'both', '--use-api', '--verbose'],
                        capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr, file=sys.stderr)



--- Step 1 ---
  Thought: [LLM] I should search
  Action: search('Earth')
  Observation: Earth is the third planet from the Sun, with a diameter of 12,742 km.

--- Step 2 ---
  Thought: [LLM] I should calculator
  Action: calc('10*5')
  Observation: 50

--- Step 3 ---
  Thought: [LLM] I should done
  Action: DONE

Final: steps=3, done=True, mode=LLM



## ReAct 的工程边界

ReAct 不是银弹。实际使用时要考虑：
- **步数限制**：无限循环 = 无限 Token 消费
- **Observation 大小**：网页全文塞进 prompt 可能超 context window
- **工具可靠性**：工具返回错误时 Agent 如何 recover
- **Termination**：Agent 何时「知道」该停了

## 练习

1. 修改上面的 `ReActAgent`：增加 `calculator` 工具。当问题包含数字时，用计算器而非搜索。
2. 实现一个 `validate()` 函数：检查 Observation 是否包含有效信息。如果无效，让 Agent 尝试不同的搜索词。
3. 设计一个场景，使 ReAct 的 loop 陷入死循环。然后修改 termination 条件来防止。

> **检查点**：你能在白板上画出 ReAct 的 Thought → Action → Observation 循环，并在每个节点标注可能出现的异常情况。